In [1]:
import pandas as pd

from get_market_cap_by_ticker import get_market_cap_by_ticker
from DATA.stock_invest_function import *
from get_fs_data_by_ticker import extract_quarterly_fs_data
from get_hscode_processed_data import get_hscode_processed_data
from get_revenue_export_joined_table import get_revenue_export_joined_table
from sarima_endog_forecast import forecast_endog_with_optional_exog
from sarima_endog_forecast import forecast_endog_fill_tail
from get_forecasted_revenue_df import build_forecast_df_from_out
from revenue_forecast_all_package import *

# 1) DB 접속정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

ticker = "A005930"
hs_code = "854232"

df_mc = get_market_cap_by_ticker(db_info, ticker)

df_rev = extract_quarterly_fs_data(
    db_info=db_info,
    table_name="korea_fs_data",          # 실제 테이블명으로 교체
    target_indicator="매출액(천원)",       # 원하는 지표명
    ticker= ticker,                     # 원하는 종목코드
)

# df_exog['monthly_raw']   # 월별 데이터
# df_exog['quarterly']     # 분기 데이터
# df_exog['exog']

df_export = get_hscode_processed_data(db_info, hs_code = hs_code)
df_exog = df_export['quarterly']

combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
    df_rev=df_rev,
    df_exog=df_exog,
    join_how="outer",
    fill_exog="ffill"   # 필요시
)


✅ A005930 시가총액 4,075건 조회 완료


In [2]:
combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
    df_rev=df_rev,
    df_exog=df_exog,
    join_how="right",
    fill_exog="ffill"   # 필요시
)


In [3]:

# combined_df: Date 인덱스 + ['endog_var','exog_var'] (exog_var는 없어도 됨)
# 예: horizon=4 (분기 4개 또는 월 4개)
out = forecast_endog_with_optional_exog(
    combined_df=final_combined_data,
    horizon=5,          # ⬅️ 예측 기간 설정
    hs_code= None,   # ⬅️ None이면 exog_var 사용 안함
    # seasonal_period=4 # 직접 지정도 가능(미지정시 자동 추론)
)
rev_forecast_with_noexog = out['forecast']

[메모리] forecast_sarima 실행 전: 448.75 MB
[메모리] find_best_sarima_params 실행 전: 448.75 MB

[메모리] find_best_sarima_params 실행 후: 453.19 MB (변화: +4.44 MB)
[메모리] forecast_sarima 실행 후: 453.25 MB (변화: +4.50 MB)


In [4]:
out2 = forecast_endog_fill_tail(final_combined_data, hs_code="854232")
rev_forecast_with_exog = out2['forecast']

[메모리] find_best_sarima_params 실행 전: 453.28 MB
[메모리] find_best_sarima_params 실행 후: 453.57 MB (변화: +0.30 MB)


In [5]:
rev_sarima_noexog = build_forecast_df_from_out(out, combined_df=final_combined_data)
rev_sarima_exog = build_forecast_df_from_out(out2, combined_df=final_combined_data)

In [6]:
# final_combined_data: get_revenue_export_joined_table(...)에서 받은 것
rev_ets_df     = forecast_revenue_ets(final_combined_data, horizon=5)
rev_prophet_df = forecast_revenue_prophet(final_combined_data, horizon=5)   # Prophet 미설치면 에러
rev_lstm_df    = forecast_revenue_lstm(final_combined_data, horizon=5, lookback=12)
rev_theta_df   = forecast_revenue_theta(final_combined_data, horizon=5)

[메모리] forecast_ets 실행 전: 453.74 MB
[메모리] forecast_ets 실행 후: 453.92 MB (변화: +0.18 MB)

22:24:00 - cmdstanpy - INFO - Chain [1] start processing



[메모리] forecast_prophet 실행 전: 453.92 MB


22:24:00 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 455.92 MB (변화: +2.00 MB)
[메모리] forecast_lstm 실행 전: 455.92 MB


[메모리] forecast_lstm 실행 후: 523.87 MB (변화: +67.95 MB)
[메모리] forecast_theta 실행 전: 523.87 MB
[메모리] forecast_theta 실행 후: 524.00 MB (변화: +0.14 MB)


In [7]:
from get_revenue_ttm_df import get_revenue_ttm_df

rev_final = get_revenue_ttm_df(
    df_rev=df_rev,
    rev_sarima_noexog=rev_sarima_noexog,
    rev_sarima_exog=rev_sarima_exog,
    rev_ets_df=rev_ets_df,
    rev_prophet_df=rev_prophet_df,
    rev_theta_df=rev_theta_df,
    rev_lstm_df=rev_lstm_df  # 또는 ref_lstm_df
    # forecast_col_map={"prophet": "yhat"}  # 필요시 예측 컬럼 강제 지정
)

### PSR 측정

In [8]:
from get_psr_from_mc_and_rev import build_psr_series
from psr_forecast_runner import forecast_psr_all_models

psr_df = build_psr_series(df_mc=df_mc, df_rev=df_rev)

In [16]:
# -------------------------------------------
# psr 예측 파이프라인 (Py3.9 호환)
# -------------------------------------------

# -------- 사용 예시 --------
psr_df = psr_df  # index=DatetimeIndex, column='psr'
exog_df = final_combined_data[['exog_var']]
horizon = 13  # 원하는 예측기간
fc_table = forecast_psr_all_models(psr_df, horizon=horizon, exog_df=exog_df)


[메모리] forecast_sarima 실행 전: 552.00 MB
[메모리] find_best_sarima_params 실행 전: 552.00 MB
[메모리] find_best_sarima_params 실행 후: 546.52 MB (변화: -5.49 MB)
[메모리] forecast_sarima 실행 후: 546.52 MB (변화: -5.49 MB)
[메모리] forecast_ets 실행 전: 546.52 MB


22:28:50 - cmdstanpy - INFO - Chain [1] start processing
22:28:50 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_ets 실행 후: 546.79 MB (변화: +0.27 MB)
[메모리] forecast_prophet 실행 전: 546.79 MB
[메모리] forecast_prophet 실행 후: 548.84 MB (변화: +2.05 MB)
[메모리] forecast_lstm 실행 전: 548.84 MB
[메모리] forecast_lstm 실행 후: 570.71 MB (변화: +21.87 MB)
[메모리] forecast_theta 실행 전: 570.71 MB
[메모리] forecast_theta 실행 후: 570.71 MB (변화: +0.00 MB)
[경고] SARIMA exog 예측 실패: exog contains inf or nans


In [34]:

# 1. 인덱스를 컬럼으로 변환
fc_table_reset = fc_table.reset_index()
rev_final_reset = rev_final.reset_index()

# 2. 'ttm'이 포함된 컬럼만 추출 (date 포함)
rev_ttm = rev_final_reset[['date'] + [col for col in rev_final_reset.columns if 'ttm' in col]]

# 3. 두 데이터프레임을 'date' 기준으로 병합
merged_df = pd.merge(
    fc_table_reset,
    rev_ttm,
    on='date',
    how='outer'   # 필요에 따라 inner, left 등 변경 가능
)

# 4. 날짜 기준 정렬
merged_df = merged_df.sort_values('date').reset_index(drop=True)

# 5. 결측값을 앞 행 값으로 채우기 (최대 2행)
merged_df = merged_df.ffill(limit=2)

# 6. 결과 확인
print(merged_df.head())

        date  psr_SARIMA_noexog  psr_SARIMA_exog  psr_ETS  psr_Prophet  \
0 2004-03-31                NaN              NaN      NaN          NaN   
1 2004-06-30                NaN              NaN      NaN          NaN   
2 2004-09-30                NaN              NaN      NaN          NaN   
3 2004-12-31                NaN              NaN      NaN          NaN   
4 2005-03-31                NaN              NaN      NaN          NaN   

   psr_LSTM  psr_Theta  revenue_sarima_ttm  revenue_sarima_exog_ttm  \
0       NaN        NaN                 NaN                      NaN   
1       NaN        NaN                 NaN                      NaN   
2       NaN        NaN                 NaN                      NaN   
3       NaN        NaN        5.763236e+10             5.763236e+10   
4       NaN        NaN        5.703091e+10             5.703091e+10   

   revenue_ets_ttm  revenue_prophet_ttm  revenue_lstm_ttm  revenue_theta_ttm  
0              NaN                  NaN          

In [36]:
merged_df.tail(15)

,date,psr_SARIMA_noexog,psr_SARIMA_exog,psr_ETS,psr_Prophet,psr_LSTM,psr_Theta,revenue_sarima_ttm,revenue_sarima_exog_ttm,revenue_ets_ttm,revenue_prophet_ttm,revenue_lstm_ttm,revenue_theta_ttm
85,2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,3.085938e+11,3.085938e+11,3.085938e+11,3.085938e+11,3.085938e+11,3.085938e+11
86,2025-09-30,NaN,NaN,NaN,NaN,NaN,NaN,3.107674e+11,3.095668e+11,3.103845e+11,3.063189e+11,3.065910e+11,3.083014e+11
87,2025-11-30,2.506973,NaN,2.548270,1.937024,2.068765,2.550677,3.107674e+11,3.095668e+11,3.103845e+11,3.063189e+11,3.065910e+11,3.083014e+11
88,2025-12-31,2.539649,NaN,2.617274,2.014040,2.167665,2.552951,3.148817e+11,3.085576e+11,3.139022e+11,3.079557e+11,3.081821e+11,3.113923e+11
89,2026-01-31,2.513915,NaN,2.626177,2.017192,2.233603,2.555224,3.148817e+11,3.085576e+11,3.139022e+11,3.079557e+11,3.081821e+11,3.113923e+11
90,2026-02-28,2.534181,NaN,2.553465,1.936445,2.279270,2.557498,3.148817e+11,3.085576e+11,3.139022e+11,3.079557e+11,3.081821e+11,3.113923e+11
91,2026-03-31,2.518221,NaN,2.596453,1.975086,2.310974,2.559771,3.182408e+11,3.068520e+11,3.136357e+11,3.068285e+11,3.073328e+11,3.074286e+11
92,2026-04-30,2.530791,NaN,2.606009,1.977678,2.330736,2.562045,3.182408e+11,3.068520e+11,3.136357e+11,3.068285e+11,3.073328e+11,3.074286e+11
93,2026-05-31,2.520891,NaN,2.554351,1.915669,2.342870,2.564318,3.182408e+11,3.068520e+11,3.136357e+11,3.068285e+11,3.073328e+11,3.074286e+11
94,2026-06-30,2.528687,NaN,2.587945,1.948164,2.340265,2.566592,3.252435e+11,3.079935e+11,3.177136e+11,3.108701e+11,3.125700e+11,3.093572e+11


In [23]:
rev_final.tail(10)

,revenue_sarima,revenue_sarima_exog,revenue_ets,revenue_prophet,revenue_lstm,revenue_theta,revenue_sarima_ttm,revenue_sarima_exog_ttm,revenue_ets_ttm,revenue_prophet_ttm,revenue_lstm_ttm,revenue_theta_ttm
date,,,,,,,,,,,,
2024-06-30,7.406830e+10,7.406830e+10,7.406830e+10,7.406830e+10,7.406830e+10,7.406830e+10,2.811685e+11,2.811685e+11,2.811685e+11,2.811685e+11,2.811685e+11,2.811685e+11
2024-09-30,7.909873e+10,7.909873e+10,7.909873e+10,7.909873e+10,7.909873e+10,7.909873e+10,2.928626e+11,2.928626e+11,2.928626e+11,2.928626e+11,2.928626e+11,2.928626e+11
2024-12-31,7.578827e+10,7.578827e+10,7.578827e+10,7.578827e+10,7.578827e+10,7.578827e+10,3.008709e+11,3.008709e+11,3.008709e+11,3.008709e+11,3.008709e+11,3.008709e+11
2025-03-31,7.914050e+10,7.914050e+10,7.914050e+10,7.914050e+10,7.914050e+10,7.914050e+10,3.080958e+11,3.080958e+11,3.080958e+11,3.080958e+11,3.080958e+11,3.080958e+11
2025-06-30,7.456632e+10,7.456632e+10,7.456632e+10,7.456632e+10,7.456632e+10,7.456632e+10,3.085938e+11,3.085938e+11,3.085938e+11,3.085938e+11,3.085938e+11,3.085938e+11
2025-09-30,8.127233e+10,8.007174e+10,8.088938e+10,7.682383e+10,7.709593e+10,7.880626e+10,3.107674e+11,3.095668e+11,3.103845e+11,3.063189e+11,3.065910e+11,3.083014e+11
2025-12-31,7.990256e+10,7.477902e+10,7.930601e+10,7.742507e+10,7.737935e+10,7.887918e+10,3.148817e+11,3.085576e+11,3.139022e+11,3.079557e+11,3.081821e+11,3.113923e+11
2026-03-31,8.249964e+10,7.743493e+10,7.887402e+10,7.801324e+10,7.829124e+10,7.517680e+10,3.182408e+11,3.068520e+11,3.136357e+11,3.068285e+11,3.073328e+11,3.074286e+11
2026-06-30,8.156900e+10,7.570780e+10,7.864424e+10,7.860794e+10,7.980351e+10,7.649493e+10,3.252435e+11,3.079935e+11,3.177136e+11,3.108701e+11,3.125700e+11,3.093572e+11


In [43]:
fc_table_reset.tail(16)

,date,psr_SARIMA_noexog,psr_SARIMA_exog,psr_ETS,psr_Prophet,psr_LSTM,psr_Theta
0,2025-11-30,2.506973,NaN,2.548270,1.937024,2.068765,2.550677
1,2025-12-31,2.539649,NaN,2.617274,2.014040,2.167665,2.552951
2,2026-01-31,2.513915,NaN,2.626177,2.017192,2.233603,2.555224
3,2026-02-28,2.534181,NaN,2.553465,1.936445,2.279270,2.557498
4,2026-03-31,2.518221,NaN,2.596453,1.975086,2.310974,2.559771
5,2026-04-30,2.530791,NaN,2.606009,1.977678,2.330736,2.562045
6,2026-05-31,2.520891,NaN,2.554351,1.915669,2.342870,2.564318
7,2026-06-30,2.528687,NaN,2.587945,1.948164,2.340265,2.566592
8,2026-07-31,2.522548,NaN,2.631782,1.993434,2.328602,2.568865
9,2026-08-31,2.527383,NaN,2.584818,1.936990,2.313945,2.571139
